# Geo-Nexus v3.2 — Build the Kaggle DAPT Dataset

## Purpose

This notebook prepares the **private Kaggle dataset used by P2 DAPT**.

It does **not** train the model.

It takes the already-created archive:

`My Drive/geonexus_v3_processed/geonexus_v3_processed.tar`

and copies **only the files required by the DAPT loader**:

- `pune_train.npy`
- `satara_train.npy`
- `pune_meta.json` (standardized from the preprocessing train-meta filename if necessary)
- `satara_meta.json` (standardized from the preprocessing train-meta filename if necessary)
- `norm_stats_trainonly.json`

Expected DAPT corpus:

- Pune train: **3315**
- Satara train: **3349**
- Total: **6664**

The notebook also verifies the files before uploading the dataset to Kaggle.


## Step 1 — Mount Google Drive

Run this cell first.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully.")


## Step 2 — Install the Kaggle client

This notebook uses the official Kaggle CLI to create the private dataset.


In [ ]:
import sys
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "kaggle"],
    text=True,
    capture_output=True,
)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("Kaggle package installation failed.")

print("Kaggle package installed successfully.")


## Step 3 — Set the source archive and output folder

The source archive is the processed dataset created during preprocessing.

**Do not change the archive path unless your file is stored somewhere else.**


In [ ]:
from pathlib import Path
import os
import shutil
import tarfile
import json
import hashlib
import numpy as np

TAR_PATH = Path(
    "/content/drive/MyDrive/geonexus_v3_processed/geonexus_v3_processed.tar"
)

DATASET_ROOT = Path("/content/geonexus-mh-v3")

REQUIRED_FILES = [
    "pune_train.npy",
    "satara_train.npy",
    "pune_meta.json",
    "satara_meta.json",
    "norm_stats_trainonly.json",
]

EXPECTED_COUNTS = {
    "pune_train.npy": 3315,
    "satara_train.npy": 3349,
}

print("Source archive:", TAR_PATH)
print("Output dataset:", DATASET_ROOT)

if not TAR_PATH.exists():
    raise FileNotFoundError(
        f"Processed archive not found: {TAR_PATH}\n"
        "Check that Google Drive is mounted and the archive exists."
    )

archive_gb = TAR_PATH.stat().st_size / (1024 ** 3)
print(f"Archive size: {archive_gb:.2f} GiB")


## Step 4 — Inspect the archive and locate the required files

This cell searches the archive by filename.

It does **not** extract the entire archive.


In [ ]:
# Find the required DAPT files anywhere inside the tar archive.
# The preprocessing notebook writes train metadata as:
#   pune_train_meta.json
#   satara_train_meta.json
# while the DAPT loader expects the standardized names:
#   pune_meta.json
#   satara_meta.json
#
# Therefore this notebook accepts either naming convention and later
# writes the standardized DAPT names into the Kaggle dataset.

ARCHIVE_ALIASES = {
    "pune_train.npy": ["pune_train.npy"],
    "satara_train.npy": ["satara_train.npy"],
    "pune_meta.json": ["pune_meta.json", "pune_train_meta.json"],
    "satara_meta.json": ["satara_meta.json", "satara_train_meta.json"],
    "norm_stats_trainonly.json": ["norm_stats_trainonly.json"],
}

with tarfile.open(TAR_PATH, mode="r") as tar:
    members = tar.getmembers()

# Build a basename -> archive-member lookup.
member_by_basename = {}
for member in members:
    if not member.isfile():
        continue
    basename = Path(member.name).name
    member_by_basename.setdefault(basename, []).append(member)

selected_members = {}

for output_name, aliases in ARCHIVE_ALIASES.items():
    matches = []

    for alias in aliases:
        matches.extend(member_by_basename.get(alias, []))

    if not matches:
        print(f"Could not find {output_name}. Searched aliases: {aliases}")

        print("\nRelevant archive members:")
        for member in members:
            basename = Path(member.name).name.lower()
            if any(token in basename for token in ("pune", "satara", "meta", "norm_stats")):
                print("  ", member.name)

        raise FileNotFoundError(
            f"Required DAPT input not found for '{output_name}'. "
            f"Searched: {aliases}"
        )

    # More than one matching file is ambiguous and should not be guessed.
    unique_names = sorted(set(member.name for member in matches))
    if len(unique_names) > 1:
        raise RuntimeError(
            f"Multiple archive members match '{output_name}': {unique_names}. "
            "Do not guess which file to use."
        )

    selected_members[output_name] = matches[0]

print("Selected archive members:")
for output_name, member in selected_members.items():
    print(f"  {output_name} <- {member.name}")

print("\nPASS: all five DAPT inputs were located.")


## Step 5 — Create a clean staging folder

Only the five required DAPT files will be copied.

Any old staging folder is removed first so that no unwanted test or annotation files remain.


In [ ]:
if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)

DATASET_ROOT.mkdir(parents=True, exist_ok=False)

print("Clean staging folder created:", DATASET_ROOT)


## Step 6 — Extract only the five DAPT files

The archive uses the preprocessing names:

- `pune_train_meta.json`
- `satara_train_meta.json`

Cell 5 already mapped those source names to the standardized DAPT names:

- `pune_meta.json`
- `satara_meta.json`

This cell uses that `selected_members` mapping directly. It does **not** look up
`pune_meta.json` or `satara_meta.json` again inside the archive.


In [ ]:
with tarfile.open(TAR_PATH, mode="r") as tar:
    for output_name in REQUIRED_FILES:
        member = selected_members[output_name]

        extracted = tar.extractfile(member)
        if extracted is None:
            raise RuntimeError(
                f"Could not read archive member: {member.name}"
            )

        # Always write the standardized filename expected by the DAPT loader.
        out_path = DATASET_ROOT / output_name

        with out_path.open("wb") as out_file:
            shutil.copyfileobj(extracted, out_file)

        print(f"Copied: {member.name} -> {output_name}")

print("Selective extraction complete.")


## Step 7 — Verify the five required files

This verifies existence and basic file readability.


In [ ]:
print("Required-file check")
print("=" * 60)

for filename in REQUIRED_FILES:
    path = DATASET_ROOT / filename

    if not path.exists():
        raise FileNotFoundError(f"Missing staged file: {filename}")

    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"{filename:28s} | {size_mb:10.2f} MiB | OK")

print("=" * 60)
print("All required files are present.")


## Step 8 — Verify the exact DAPT corpus size

The architecture-aligned DAPT corpus is:

**3315 Pune + 3349 Satara = 6664 pairs**

This check must pass before the dataset is uploaded.


In [ ]:
pune = np.load(DATASET_ROOT / "pune_train.npy", mmap_mode="r")
satara = np.load(DATASET_ROOT / "satara_train.npy", mmap_mode="r")

print("Pune train shape  :", pune.shape)
print("Satara train shape:", satara.shape)

actual_pune = len(pune)
actual_satara = len(satara)
actual_total = actual_pune + actual_satara

print()
print("Pune train   :", actual_pune)
print("Satara train :", actual_satara)
print("Total DAPT   :", actual_total)

assert actual_pune == EXPECTED_COUNTS["pune_train.npy"], (
    f"Pune count mismatch: expected {EXPECTED_COUNTS['pune_train.npy']}, "
    f"got {actual_pune}"
)
assert actual_satara == EXPECTED_COUNTS["satara_train.npy"], (
    f"Satara count mismatch: expected {EXPECTED_COUNTS['satara_train.npy']}, "
    f"got {actual_satara}"
)
assert actual_total == 6664, (
    f"DAPT corpus mismatch: expected 6664, got {actual_total}"
)

print()
print("PASS: DAPT corpus contains exactly 6664 pairs.")


## Step 9 — Verify the metadata files and train-only normalization

The DAPT loader expects the metadata files and `norm_stats_trainonly.json`.

We also check that the normalization file contains the expected training statistics structure.


In [ ]:
for filename in ["pune_meta.json", "satara_meta.json"]:
    with (DATASET_ROOT / filename).open("r", encoding="utf-8") as f:
        meta = json.load(f)

    if not isinstance(meta, list):
        raise TypeError(f"{filename} is not a JSON list.")
    print(f"{filename}: {len(meta)} metadata records")

with (DATASET_ROOT / "norm_stats_trainonly.json").open("r", encoding="utf-8") as f:
    norm_stats = json.load(f)

required_norm_keys = ["mean", "std", "array_scale", "sar_channels", "sar_extra_scale"]
missing_norm_keys = [k for k in required_norm_keys if k not in norm_stats]

if missing_norm_keys:
    raise KeyError(
        "norm_stats_trainonly.json is missing keys: "
        + ", ".join(missing_norm_keys)
    )

print("norm_stats_trainonly.json: required keys present")
print("PASS: metadata and normalization file are readable.")


## Step 10 — Verify the final staging folder

Only these files should be present:

```text
pune_train.npy
satara_train.npy
pune_meta.json
satara_meta.json
norm_stats_trainonly.json
```

The Kaggle CLI metadata file is added in the next step.


In [ ]:
staged_files = sorted(
    p.name for p in DATASET_ROOT.iterdir()
    if p.is_file()
)

print("Files currently staged:")
for name in staged_files:
    print("  ", name)

unexpected = sorted(set(staged_files) - set(REQUIRED_FILES))

if unexpected:
    raise RuntimeError(
        "Unexpected files found in the DAPT dataset staging folder: "
        + ", ".join(unexpected)
    )

print("PASS: no unwanted data files are staged.")


## Step 11 — Configure Kaggle API

Upload your `kaggle.json` API token when prompted.

You can create it in Kaggle:

**Profile → Settings → API → Create New Token**


In [ ]:
from google.colab import files

uploaded = files.upload()

if "kaggle.json" not in uploaded:
    raise FileNotFoundError(
        "Please upload the Kaggle API file named 'kaggle.json'."
    )

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

kaggle_json_path = kaggle_dir / "kaggle.json"
shutil.copy2("kaggle.json", kaggle_json_path)
os.chmod(kaggle_json_path, 0o600)

print("Kaggle API credentials installed.")


## Step 12 — Create `dataset-metadata.json`

The Kaggle username is read automatically from `kaggle.json`, so there is no hard-coded username.


In [ ]:
with open(kaggle_json_path, "r", encoding="utf-8") as f:
    kaggle_credentials = json.load(f)

kaggle_username = kaggle_credentials.get("username")

if not kaggle_username:
    raise KeyError(
        "The uploaded kaggle.json does not contain a 'username' field."
    )

DATASET_SLUG = "geonexus-mh-v3"
DATASET_ID = f"{kaggle_username}/{DATASET_SLUG}"

metadata = {
    "title": "Geo-Nexus Maharashtra CD v3.2 — DAPT Train Data",
    "id": DATASET_ID,
    "licenses": [
        {"name": "CC-BY-SA-4.0"}
    ]
}

metadata_path = DATASET_ROOT / "dataset-metadata.json"

with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Kaggle dataset ID:", DATASET_ID)
print("Metadata written to:", metadata_path)


## Step 13 — Final pre-upload integrity check

Do not upload unless this cell passes.


In [ ]:
final_files = sorted(
    p.name for p in DATASET_ROOT.iterdir()
    if p.is_file()
)

expected_final_files = sorted(REQUIRED_FILES + ["dataset-metadata.json"])

print("Final files:")
for name in final_files:
    print("  ", name)

assert final_files == expected_final_files, (
    "Final dataset contents do not match the expected file list.\n"
    f"Expected: {expected_final_files}\n"
    f"Found:    {final_files}"
)

# Re-check the DAPT counts one more time.
pune = np.load(DATASET_ROOT / "pune_train.npy", mmap_mode="r")
satara = np.load(DATASET_ROOT / "satara_train.npy", mmap_mode="r")

assert len(pune) == 3315
assert len(satara) == 3349
assert len(pune) + len(satara) == 6664

print()
print("FINAL CHECK: PASS")
print("Files: PASS")
print("Pune: 3315")
print("Satara: 3349")
print("Total: 6664")
print("Ready for Kaggle upload.")


## Step 14 — Create the Kaggle dataset

This creates a **private Kaggle dataset** named `geonexus-mh-v3`.

Run this cell **once** for the first creation.

If Kaggle reports that the dataset already exists, use Step 15 instead.


In [ ]:
import subprocess
import sys

create_cmd = [
    sys.executable,
    "-m",
    "kaggle",
    "datasets",
    "create",
    "-p",
    str(DATASET_ROOT),
    "--dir-mode",
    "zip",
]

print("Creating Kaggle dataset...")
result = subprocess.run(
    create_cmd,
    text=True,
    capture_output=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(
        "Kaggle dataset creation failed. "
        "If the dataset already exists, continue to Step 15."
    )

print("Kaggle dataset created successfully.")
print("Dataset:", DATASET_ID)


## Step 15 — Update an existing dataset instead of creating a duplicate

Run this **only if Step 14 says the dataset already exists**.


In [ ]:
update_cmd = [
    sys.executable,
    "-m",
    "kaggle",
    "datasets",
    "version",
    "-p",
    str(DATASET_ROOT),
    "-m",
    "Geo-Nexus v3.2 DAPT dataset — 6664 pairs",
    "--dir-mode",
    "zip",
]

print("Updating existing Kaggle dataset...")
result = subprocess.run(
    update_cmd,
    text=True,
    capture_output=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Kaggle dataset update failed.")

print("Kaggle dataset version uploaded successfully.")
print("Dataset:", DATASET_ID)


# Final result

After the upload, attach these **two datasets** to the Kaggle DAPT notebook:

1. **Geo-Nexus Maharashtra CD v3.2 — DAPT Train Data**
   - mounts as `/kaggle/input/geonexus-mh-v3/`

2. **Geo-Nexus v3.2 Pretrained Foundation Weights**
   - mounts as `/kaggle/input/ssl4eo-weights/`

The DAPT notebook should then see:

```text
/kaggle/input/geonexus-mh-v3/
    pune_train.npy
    satara_train.npy
    pune_meta.json
    satara_meta.json
    norm_stats_trainonly.json

/kaggle/input/ssl4eo-weights/
    resnet18_s2c_moco.pth
    resnet18_s1_bigearthnet.pth
```

**Do not start DAPT training from this notebook.**

After the Kaggle dataset is created, open `02_dapt_FINAL_v3_2_6664.ipynb` and run its verification cells first.
